<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Imports](#1)
- [2. Data](#P2)
- [3. General Text Preprocessing](#3)
    - [3.1 Clean Data](#3_1)
    - [3.2 Transform Data](#3_2)
    - [3.3 Normalize Data](#3_3)
    - [3.4 Creating the POS tags](#3_4)


# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)


In [2]:
# !pip install deep-translator
# !pip install langdetect
# !pip install langid
# !pip install emoji

In [3]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [ ]:
import sys
import os
import pandas as pd
import regex as re
from collections import Counter
import Levenshtein
from jellyfish import jaro_winkler_similarity

# Add the source folder to sys.path so we can import our modules
for path in ['../source', './source']:
    source_code_path = os.path.abspath(path)
    if os.path.exists(source_code_path):
        if source_code_path not in sys.path:
            sys.path.append(source_code_path)
        break

from my_utils import *
from general_preprocessing import *
from visualizations import *


# <font color='#BFD72F' size=6>**2. Data**</font> <a class="anchor" id="P2"></a>

[Back to TOC](#toc)


In [5]:
dataset_original = load_dataset('../data/GoEmotions.csv')

In [6]:
dataset_original.head()

,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [7]:
dataset_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211225 entries, 0 to 211224
Data columns (total 37 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   text                  211225 non-null  object 
 1   id                    211225 non-null  object 
 2   author                211225 non-null  object 
 3   subreddit             211225 non-null  object 
 4   link_id               211225 non-null  object 
 5   parent_id             211225 non-null  object 
 6   created_utc           211225 non-null  float64
 7   rater_id              211225 non-null  int64  
 8   example_very_unclear  211225 non-null  bool   
 9   admiration            211225 non-null  int64  
 10  amusement             211225 non-null  int64  
 11  anger                 211225 non-null  int64  
 12  annoyance             211225 non-null  int64  
 13  approval              211225 non-null  int64  
 14  caring                211225 non-null  int64  
 15  

In [8]:
dataset = dataset_original.copy()

# <font color='#BFD72F' size=6>**3. General Text Preprocessing**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)


1. **Clean Data** — Remove Extraneous Content

2. **Transform Data** — Tokenization, POS Tagging, Stopword Removal

3. **Normalize Data** — Case, Stemming, Lemmatization


## <font color='#BFD72F' size=6>3.1 Clean Data</font> <a class="anchor" id="3_1"></a>

[Back to TOC](#toc)


**Remove Extraneous Content (Emojis and HTML/Web elements)**


we have emojis in the text, so we need to remove them before doing any preprocessing. We can use the emoji library to do this.

In [9]:
# Just get the indices of rows with emojis
emoji_indices = dataset[dataset['text'].astype(str).apply(
    lambda x: any(char in emoji.EMOJI_DATA for char in x)
)].index.tolist()

print(f"Found {len(emoji_indices)} rows with emojis")

# View first few
print(dataset.loc[emoji_indices[:10], ['text']])

Found 4246 rows with emojis
                                                  text
102  Nice! You and her would get along i bet. She l...
132  Funny, the right only seems to say this about ...
168                          I love this sub reddit 😍😂
246   That’s what’s it’s called or what SHE’S called?🤣
260  Looks like complete slop with 4 turds thrown o...
271  😭 I wanted her to be on the top 4; now we’ve h...
279  From what the experts say his chances are like...
283  Well they called and said there was a pricing ...
284  It's the great irony of The Resistance©. It's ...
315      Ha! They’re very naughty! Pretty cute, tho ❤️


In [10]:
len(emoji_indices)

4246

In [11]:
print(f"Row indices with emojis: {emoji_indices[:20]}")  # First 20

Row indices with emojis: [102, 132, 168, 246, 260, 271, 279, 283, 284, 315, 340, 342, 416, 654, 818, 1012, 1039, 1050, 1065, 1102]


In [12]:
dataset[['text']].iloc[emoji_indices[10:20]]

,text
340,I'm so fucking dead!!!!!!!💀💀💀💀🤣🤣🤣🤣🤣😂🤣🤣🤣🤣
342,“Stop. What are you doing? Stop.” 😂
416,Looks gorgeous on u 😍😍
654,He did get 3 stars bro😭
818,You have another new follower 😊 love your work!
1012,GG haha 😉
1039,sorry [NAME]! 😘😘😘
1050,I’m so glad someone linked it. You’re doing th...
1065,Amazing doggo 😍
1102,I'd punt her to another planet if she said tha...


Extracting the emojis, translating them into text so that we can use them further along for our analysis. We will keep the original text with emojis for now, and we will create a new column with the translated text without emojis. This way we can compare the two and see if there are any differences in the results.

In [13]:
# Part 1
dataset['01_cleaned_text'] = dataset['text'].apply(
    lambda x: main_pipeline(
        print_output=False,
        raw_text=x,
        no_emojis=True,
        no_hashtags=False,
        hashtag_retain_words=True,
        no_newlines=False,
        no_urls=False,
        no_punctuation=False,
        no_stopwords=False,
        custom_stopwords=[],
        stopwords_tokeep=[],
        convert_diacritics=False,
        lowercase=False,
        lemmatized=False,
        list_pos=[],
        pos_tags_list='no_pos',
        tokenized_output=False,
        stemmed=False,
        treat_repeated_chars=False
    )
)

In [14]:
dataset[['text', '01_cleaned_text']].iloc[emoji_indices[10:20]]

,text,01_cleaned_text
340,I'm so fucking dead!!!!!!!💀💀💀💀🤣🤣🤣🤣🤣😂🤣🤣🤣🤣,I am so fucking dead!!!!!!! emoji_skullemoji_s...
342,“Stop. What are you doing? Stop.” 😂,“ Stop . What are you doing? Stop. ” emoji_fac...
416,Looks gorgeous on u 😍😍,Looks gorgeous on u emoji_smiling_face_with_he...
654,He did get 3 stars bro😭,He did get 3 stars broemoji_loudly_crying_face
818,You have another new follower 😊 love your work!,You have another new follower emoji_smiling_fa...
1012,GG haha 😉,GG haha emoji_winking_face
1039,sorry [NAME]! 😘😘😘,sorry [NAME]! emoji_face_blowing_a_kissemoji_f...
1050,I’m so glad someone linked it. You’re doing th...,I ’ m so glad someone linked it . You ’ re doi...
1065,Amazing doggo 😍,Amazing doggo emoji_smiling_face_with_heart-eyes
1102,I'd punt her to another planet if she said tha...,I would punt her to another planet if she said...


## <font color='#BFD72F' size=6>3.2 Transform Data</font> <a class="anchor" id="3_2"></a>

[Back to TOC](#toc)


**Tokenization, POS Tagging, Stopword Removal**


In [15]:
# Part 3
dataset['02_transformed_text'] = dataset['01_cleaned_text'].apply(
    lambda x: main_pipeline(
        raw_text=x,
        no_emojis=False,             
        no_hashtags=False,           
        hashtag_retain_words=True,  
        no_newlines=False,           
        no_urls=False,
        no_punctuation=False,  
        no_stopwords=True,
        custom_stopwords=[],
        stopwords_tokeep=[],
        convert_diacritics=False,
        lowercase=False,
        lemmatized=False,
        list_pos=[], #'n', 'v', 'a', 'r', 's'
        pos_tags_list='no_pos',
        tokenized_output=True,
        stemmed=False,
        treat_repeated_chars=False
    )
)

## <font color='#BFD72F' size=6>3.3 Normalize Data</font> <a class="anchor" id="3_3"></a>

[Back to TOC](#toc)


**Case, Stemming, Lemmatization**


In [16]:
# Part 3
dataset['03_normalized_text'] = dataset['02_transformed_text'].apply(
    lambda x: main_pipeline(
        raw_text=x,
        no_emojis=False,             
        no_hashtags=False,           
        hashtag_retain_words=True,  
        no_newlines=False,           
        no_urls=False,
        no_punctuation=True,  
        no_stopwords=False,
        custom_stopwords=[],
        stopwords_tokeep=[],
        convert_diacritics=False,
        lowercase=True,
        lemmatized=True,
        list_pos=['n', 'v', 'a', 'r', 's'],
        pos_tags_list='pos_tuples',
        tokenized_output=True,
        stemmed=False,
        treat_repeated_chars=False
    )
)

In [17]:
# Follow an example
print("Original Text Sample:\n")
print(dataset[['text']].iloc[818].values)

print("\nCleaned Text Sample:\n")
print(dataset[['01_cleaned_text']].iloc[818].values)

print("\nTransformed Text Sample:\n")
print(dataset[['02_transformed_text']].iloc[818].values)

print("\nNormalized Text Sample:\n")
print(dataset[['03_normalized_text']].iloc[818].values)

Original Text Sample:

['You have another new follower 😊 love your work!']

Cleaned Text Sample:

['You have another new follower emoji_smiling_face_with_smiling_eyes love your work!']

Transformed Text Sample:

[list(['another', 'new', 'follower', 'emoji_smiling_face_with_smiling_eyes', 'love', 'work', '!'])]

Normalized Text Sample:

[list([("'another", 'PRP$'), ('new', 'JJ'), ('follower', 'JJR'), ('emoji_smiling_face_with_smiling_eyes', 'NNS'), ('love', 'VBP'), ('work', 'NN'), ("'", "''")])]


In [18]:
dataset['token_count'] = dataset['02_transformed_text'].apply(len)

histogram_chart(
    data=dataset,
    column='token_count',
    title='Token Count per Review',
    x_label='Number of Tokens'
)

In [19]:
dataset['review_char_len'] = dataset['text'].astype(str).str.len()

histogram_chart(
    data=dataset,
    column='review_char_len',
    title='Review Length (Characters)',
    x_label='Number of Characters'
)